# Questão 6 - Previsão de Valor de Imóveis (Nível Avançado)
Pipeline completo com Feature Engineering, múltiplos modelos, tuning e análise.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')


In [ ]:
df = pd.read_csv('california_housing.csv')
df.head()


In [ ]:
df.info()
df.describe()


In [ ]:
# Feature Engineering
df['Rooms_per_Household'] = df['AveRooms'] / df['AveOccup']
df['Bedrooms_per_Room'] = df['AveBedrms'] / df['AveRooms']
df['Population_per_Household'] = df['Population'] / df['AveOccup']
df.head()


In [ ]:
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
numeric_features = X.columns.tolist()
numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])
preprocessor = ColumnTransformer(transformers=[('num', numeric_transformer, numeric_features)])


In [ ]:
# Pipelines de modelos
pipe_lr = Pipeline(steps=[('pre', preprocessor), ('model', LinearRegression())])
pipe_rf = Pipeline(steps=[('pre', preprocessor), ('model', RandomForestRegressor(random_state=42))])
pipe_gb = Pipeline(steps=[('pre', preprocessor), ('model', GradientBoostingRegressor(random_state=42))])
pipe_mlp = Pipeline(steps=[('pre', preprocessor), ('model', MLPRegressor(random_state=42, max_iter=500))])


In [ ]:
models = {
    'LinearRegression': pipe_lr,
    'RandomForest': pipe_rf,
    'GradientBoosting': pipe_gb,
    'MLPRegressor': pipe_mlp
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    results[name] = {'RMSE': rmse, 'R2': r2}
results


In [ ]:
# Tuning Random Forest
param_grid_rf = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20]
}
grid_rf = GridSearchCV(pipe_rf, param_grid_rf, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
grid_rf.fit(X_train, y_train)
best_rf = grid_rf.best_estimator_
preds_rf = best_rf.predict(X_test)
rmse_rf = np.sqrt(mean_squared_error(y_test, preds_rf))
r2_rf = r2_score(y_test, preds_rf)
rmse_rf, r2_rf, grid_rf.best_params_


In [ ]:
# Feature Importance (Random Forest)
rf_model = best_rf.named_steps['model']
importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)
feat_imp


## Conclusão
- Compare os RMSE e R².
- O melhor modelo é o que tiver menor RMSE e maior R².
- Cite as variáveis mais importantes.
- Sugestões de melhoria: mais dados, tuning, feature engineering.
